In [ ]:
!pip install langdetect backoff

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=1322c87a46f0144d3ff2617e2143152cfbd4bcbb4419bf1dcd503f9a68b2b79a
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


#IMPORTING AND LOGGING

In [ ]:
import requests
import pandas as pd
import time
import json
import re
import logging
import backoff
from langdetect import detect, LangDetectException

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

# CONFIGURATION

In [ ]:
CONFIG = {
    "start_year": 2010,
    "end_year": 2025,
    "results_per_keyword": 100,  # Number of results to request per keyword
    "min_abstract_length": 200, # Minimum abstract length in characters to filter out snippets & placeholders
    "allowed_fields": ["Environmental Science", "Geology", "Geography", "Biology"],  # Only keeping papers from these field
    "language": "en", # Only keeping English abstracts

    "output_file": "climate_abstracts.csv",
    "api_key": None,
}

# KEYWORDS

In [ ]:
KEYWORDS = [
    "climate change impacts ecosystem",
    "CO2 emissions global warming",
    "sea level rise coastal flooding",
    "glacier melt arctic temperature",
    "deforestation carbon emissions",
    "coral bleaching ocean warming",
    "drought wildfire climate change",
    "greenhouse gas atmosphere feedback",
    "biodiversity loss climate change",
    "renewable energy carbon reduction",
    "methane emissions permafrost",
    "ocean acidification marine species",
    "extreme weather events climate",
    "carbon cycle terrestrial ecosystem",
    "Arctic ice loss climate feedback",
]


# TEXT CLEANING

In [ ]:
def standardize_chemical_formulas(text: str) -> str:
    replacements = {
        # LaTeX-style subscripts
        r'CO[\s_]*\{?2\}?': 'CO2',
        r'CH[\s_]*\{?4\}?': 'CH4',
        r'N[\s_]*\{?2\}?O': 'N2O',
        r'O[\s_]*\{?3\}?': 'O3',
        r'NO[\s_]*\{?2\}?': 'NO2',
        r'SO[\s_]*\{?2\}?': 'SO2',
        r'H[\s_]*\{?2\}?O': 'H2O',
        # Temperature symbols
        r'°\s*C': '°C',
        r'°\s*F': '°F',
        r'℃': '°C',
        # Common LaTeX artifacts
        r'\$([^$]+)\$': r'\1',   # removing $ math delimiters
        r'\\[a-zA-Z]+\{([^}]+)\}': r'\1',  # removing \command{text}
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
    return text


def clean_abstract(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = standardize_chemical_formulas(text)  # Standardizing chemical formulas & symbols
    text = re.sub(r'<[^>]+>', '', text)  #Removing HTML tags if any
    text = re.sub(r'\s+', ' ', text).strip() #Collapsing multiple whitespace / newlines into single space
    text = re.sub(r'[^\x20-\x7E°²³]', '', text)    #Removing non-printable characters

    return text


def is_english(text: str) -> bool:
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False


def is_valid_abstract(text: str, min_length: int = 200) -> bool:
    """Return True if abstract passes quality checks."""
    if not text or len(text) < min_length:
        return False
    # Filter out placeholder abstracts
    placeholders = [
        "abstract not available",
        "no abstract",
        "not available",
        "abstract missing",
    ]
    lower = text.lower()
    if any(p in lower for p in placeholders):
        return False
    return True

# API FETCHING

In [ ]:
def build_headers(api_key=None):
    headers = {"Accept": "application/json"}
    if api_key:
        headers["x-api-key"] = api_key
    return headers


@backoff.on_exception(
    backoff.expo,
    requests.exceptions.HTTPError,
    max_tries=5,            # Retry up to 5 times
    giveup=lambda e: e.response is not None and e.response.status_code not in [429, 500, 503],
    on_backoff=lambda details: logger.warning(
        f"Rate limited. Retrying in {details['wait']:.1f}s... (attempt {details['tries']})"
    )
)
def _api_request(url, params, headers):
    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()  # Raises HTTPError for 4xx/5xx
    return response.json()


def fetch_abstracts(query: str, limit: int = 100, api_key: str = None) -> list:
    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {
        "query": query,
        "limit": min(limit, 100),  # API hard cap is 100 per request
        "fields": (
            "title,"
            "abstract,"
            "year,"
            "authors,"
            "venue,"
            "citationCount,"       #  For influence ranking in KG analytics
            "s2FieldsOfStudy"      #  For domain filtering
        )
    }

    headers = build_headers(api_key)

    try:
        data = _api_request(url, params, headers)
        papers = data.get("data", [])
        logger.info(f"Fetched {len(papers)} papers for: '{query}'")
        return papers
    except requests.exceptions.HTTPError as e:
        logger.error(f"Failed after retries for '{query}': {e}")
        return []
    except Exception as e:
        logger.error(f" Unexpected error for '{query}': {e}")
        return []


# FIELD OF STUDY FILTER

In [ ]:
def is_relevant_field(paper: dict, allowed_fields: list) -> bool:
    """
    Return True if paper belongs to at least one allowed field of study.
    If allowed_fields is None, always return True.
    """
    if not allowed_fields:
        return True

    fields = paper.get("s2FieldsOfStudy", []) or []
    paper_fields = [f.get("category", "") for f in fields]

    return any(af in paper_fields for af in allowed_fields)

# MAIN PIPELINE

In [ ]:
def run_pipeline(keywords: list, config: dict) -> pd.DataFrame:
    all_papers = []
    logger.info(f"Starting collection for {len(keywords)} keywords...")
    logger.info("=" * 60)

    for i, keyword in enumerate(keywords, 1):
        logger.info(f"[{i}/{len(keywords)}] Querying: '{keyword}'")

        papers = fetch_abstracts(
            query=keyword,
            limit=config["results_per_keyword"],
            api_key=config["api_key"]
        )

        if config["allowed_fields"]: # Filter by field of study
            before = len(papers)
            papers = [p for p in papers if is_relevant_field(p, config["allowed_fields"])]
            logger.info(f"  → Field filter: {before} → {len(papers)} papers kept")

        all_papers.extend(papers)

        time.sleep(1.5) # Polite delay between requests

    logger.info("=" * 60)
    logger.info(f"Total raw papers collected: {len(all_papers)}")

    #  Converting to DataFrame
    records = []
    for p in all_papers:
        # Flattening authors list to a comma-separated string
        authors = ", ".join(
            [a.get("name", "") for a in (p.get("authors") or [])]
        )
        # Flattening fields of study
        fields = ", ".join(
            [f.get("category", "") for f in (p.get("s2FieldsOfStudy") or [])]
        )
        records.append({
            "title":          p.get("title", ""),
            "abstract":       p.get("abstract", ""),
            "year":           p.get("year", None),
            "authors":        authors,
            "venue":          p.get("venue", ""),
            "citation_count": p.get("citationCount", 0),
            "fields":         fields,
        })

    df = pd.DataFrame(records)
    logger.info(f"After conversion: {len(df)} rows")

    # Removing missing abstracts
    df = df[df["abstract"].notna() & (df["abstract"] != "")]
    logger.info(f"After removing empty abstracts: {len(df)} rows")

    # Cleaning abstract text
    logger.info("Cleaning abstracts...")
    df["abstract"] = df["abstract"].apply(clean_abstract)

    # Filtering short / placeholder abstracts
    df = df[df["abstract"].apply(
        lambda x: is_valid_abstract(x, config["min_abstract_length"])
    )]
    logger.info(f"After length/placeholder filter: {len(df)} rows")

    # Language detection
    logger.info("Detecting language (keeping English only)...")
    df["is_english"] = df["abstract"].apply(is_english)
    df = df[df["is_english"]].drop(columns=["is_english"])
    logger.info(f"After language filter: {len(df)} rows")

    #Deduplicate by title
    df = df.drop_duplicates(subset="title").reset_index(drop=True)
    logger.info(f"After deduplication: {len(df)} rows")

    logger.info("=" * 60)
    logger.info(f"Final clean dataset: {len(df)} abstracts")

    return df

# RUN AND SAVE

In [ ]:
df = run_pipeline(KEYWORDS, CONFIG)

# Preview
print("\nSample of collected data:")
print(df[["title", "year", "citation_count", "fields"]].head(10).to_string())

# Save
df.to_csv(CONFIG["output_file"], index=False)
logger.info(f"\nSaved to: {CONFIG['output_file']}")

# Basic stats
print(f"\nDataset Summary:")
print(f"   Total abstracts : {len(df)}")
print(f"   Year range      : {df['year'].min()} – {df['year'].max()}")
print(f"   Avg citations   : {df['citation_count'].mean():.1f}")
print(f"   Top fields      :\n{df['fields'].value_counts().head(5)}")

INFO:backoff:Backing off _api_request(...) for 1.0s (requests.exceptions.HTTPError: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=climate+change+impacts+ecosystem&limit=100&fields=title%2Cabstract%2Cyear%2Cauthors%2Cvenue%2CcitationCount%2Cs2FieldsOfStudy)
INFO:backoff:Backing off _api_request(...) for 1.9s (requests.exceptions.HTTPError: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=climate+change+impacts+ecosystem&limit=100&fields=title%2Cabstract%2Cyear%2Cauthors%2Cvenue%2CcitationCount%2Cs2FieldsOfStudy)
INFO:backoff:Backing off _api_request(...) for 3.0s (requests.exceptions.HTTPError: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=climate+change+impacts+ecosystem&limit=100&fields=title%2Cabstract%2Cyear%2Cauthors%2Cvenue%2CcitationCount%2Cs2FieldsOfStudy)
INFO:backoff:Backing off _api_request(...) for 5.0s (requests.exceptions.HTTPError: 429 Client Error:  


Sample of collected data:
                                                                                                                                                                   title    year  citation_count                                                  fields
0                                            Episodic vs. Sea Level Rise Coastal Flooding Scenarios at the Urban Scale: Extreme Event Analysis and Adaptation Strategies  2025.0               0                        Environmental Science, Geography
1                                                             Future Coastal Population Growth and Exposure to Sea-Level Rise and Coastal Flooding - A Global Assessment  2015.0            2301   Medicine, Geography, Environmental Science, Geography
2                                                                     New elevation data triple estimates of global vulnerability to sea-level rise and coastal flooding  2019.0            1037  Environmental Science, Medicine,

# LOCAL SAVING

In [ ]:
from google.colab import files
files.download(CONFIG["output_file"])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>